# Tutorial 2: Sending Selected Runs to Remote Server

## 🎯 Learning Objectives

By the end of this tutorial, you will:
- Understand how to point MLflow at a remote server instead of your local one
- Learn how to send only your best run(s) to the remote server, without re-deriving them from local `mlruns/`/`mlartifacts/`
- Know what extra artifacts (preprocessing code, dependencies) a remote/production server needs beyond just the model

---

## 📋 Prerequisites

- Completed **Tutorial 1** (local MLflow setup)
- Access credentials to the remote MLflow server (username + password)

---

## 🔑 Key Concept: Why Re-run Instead of Transfer?

MLflow doesn't have a "copy run" feature. Instead, we:

1. **Experiment freely locally** (unlimited runs, no server load)
2. **Identify the best model** (using the local UI/metrics)
3. **Re-run ONLY that model** on the remote server, by keeping the same data pipeline and hyperparameters and just switching the tracking URI

This approach:
-  Keeps remote servers clean and organized
-  Reduces bandwidth and storage costs
-  Ensures reproducibility
-  Maintains complete experiment history locally

---


## 1. Setting Up Remote Server Credentials

Before connecting to the remote server, you need to set up authentication.

### 🔐 Set environment variables

In your terminal (or add to your `.bashrc`/`.zshrc`):

```bash
export MLFLOW_TRACKING_USERNAME="your_username"
export MLFLOW_TRACKING_PASSWORD="your_password"
export MLFLOW_TRACKING_URI="https://mlflow-dev.fink-broker.org"
```

⚠️ **Security Note**: Never hardcode credentials in your notebooks or scripts!

---


### Verify environment variables are set


In [1]:
import os

# Check if credentials are set
required_vars = ["MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD", "MLFLOW_TRACKING_URI"]

for var in required_vars:
    if var in os.environ:
        print(f"{var} is set")
    else:
        print(f"{var} is NOT set - please set it before continuing!")
        
# Display the tracking URI (safe to show)
if "MLFLOW_TRACKING_URI" in os.environ:
    print(f"\n🔗 Remote server: {os.environ['MLFLOW_TRACKING_URI']}")


MLFLOW_TRACKING_USERNAME is NOT set - please set it before continuing!
MLFLOW_TRACKING_PASSWORD is NOT set - please set it before continuing!
MLFLOW_TRACKING_URI is NOT set - please set it before continuing!


## 2. Reuse your data pipeline and best parameters

Once you're happy with a run in the local MLflow UI, the simplest way to get it onto the remote server is: **keep the exact same data pipeline and hyperparameters, and just point MLflow at the remote server before logging.** No need to dig through `mlruns/`/`mlartifacts/` on disk or look up a specific run ID — just rebuild `X`/`y` the same way as in Tutorial 1, and reuse the parameters that gave you the best local run.

In [ ]:
import pandas as pd
import numpy as np
import preprocessing as pp
from importlib import resources

with resources.path("data", "test_alerts.parquet") as parquet_path:
    PARQUET_FILE = parquet_path

alerts_df = pd.read_parquet(PARQUET_FILE)
cut_alerts_df = pp.make_cut(alerts_df)
clean_df = pp.raw2clean(cut_alerts_df)
curated_df = pp.run_sherlock(clean_df)
X, ids = pp.make_X(curated_df)
X = X.dropna(axis=1, how="all")

# Same mock labels as Tutorial 1 (use your real labels here)
y = np.array([0] * X.shape[0])
y = pd.DataFrame(y, columns=["labels"])

print(f"X shape: {X.shape}, y shape: {y.shape}")

In [ ]:
# Reuse whichever hyperparameters gave you the best local run
# (check the local MLflow UI at http://127.0.0.1:6969 if you don't remember them)
PARAMS = {
    "learning_rate": 0.1,
    "random_state": 42
}
print(f"Parameters: {PARAMS}")

## 3. Generate Environment Dependencies

The remote server needs to know which packages are required to run your preprocessing code.


In [ ]:
# Preprocessing dependencies (see the imports at the top of preprocessing.py)
dependencies = ["pandas", "numpy", "fink-client", "lasair"]
dependencies_path = "requirements.txt"
with open(dependencies_path, "w") as f:
    f.write("\n".join(dependencies) + "\n")

print(f"Dependencies written to {dependencies_path}:")
print(open(dependencies_path).read())

## 4. Re-run on Remote Server

Now we have everything we need! Let's re-run the experiment on the remote server.

### 🌐 Switch to remote tracking URI


In [ ]:
import mlflow
from mlflow.tracking import MlflowClient
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from mlflow.models import infer_signature
import json
import getpass

# Get remote server URL and credentials.
# Environment variables are used first (see section 1); if any are missing,
# you'll be prompted interactively. Never hardcode credentials here.
REMOTE_URI = (
    os.environ.get("MLFLOW_TRACKING_URI")
    or input("MLFLOW_TRACKING_URI [https://mlflow-dev.fink-broker.org]: ")
    or "https://mlflow-dev.fink-broker.org"
)
username = os.environ.get("MLFLOW_TRACKING_USERNAME") or input("MLFLOW_TRACKING_USERNAME: ")
password = os.environ.get("MLFLOW_TRACKING_PASSWORD") or getpass.getpass("MLFLOW_TRACKING_PASSWORD: ")

if username:
    os.environ["MLFLOW_TRACKING_USERNAME"] = username
if password:
    os.environ["MLFLOW_TRACKING_PASSWORD"] = password

print(f"Remote: {REMOTE_URI}")

# Switch to remote server
mlflow.set_tracking_uri(REMOTE_URI)
mlflow.set_experiment("tutorial")  # Use same experiment name as Tutorial 1

client = MlflowClient()

print("Connected to remote server!")

### 🚀 Run the experiment on remote server

This is almost identical to Tutorial 1, but with TWO critical additions for the remote server:
1. Log the preprocessing code
2. Log the requirements.txt


In [ ]:
print("Starting run on REMOTE server...\n")

with mlflow.start_run(run_name=f"remote_LR_{PARAMS['learning_rate']}"):

    # ==========================================
    # 1. TRAIN MODEL (same as before)
    # ==========================================
    print("Training model...")
    model = HistGradientBoostingClassifier(**PARAMS)
    model.fit(X.values, y)
    y_pred = model.predict(X.values)
    print("Model trained!\n")

    # ==========================================
    # 2. LOG PARAMETERS (same as before)
    # ==========================================
    print("Logging parameters...")
    mlflow.log_params(PARAMS)

    # ==========================================
    # 3. LOG MODEL (same as before)
    # ==========================================
    print("Logging model...")
    signature = infer_signature(X, y_pred)
    mlflow.sklearn.log_model(
        model,
        name="model",
        signature=signature,
        input_example=X.iloc[:1],
    )

    # ==========================================
    # 4. LOG METRICS (same as before)
    # ==========================================
    print("Logging metrics...")
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))
    mlflow.log_metric("precision", precision_score(y, y_pred, zero_division=0))
    mlflow.log_metric("recall", recall_score(y, y_pred, zero_division=0))
    mlflow.log_metric("f1_score", f1_score(y, y_pred, zero_division=0))

    # ==========================================
    # 5. LOG DATA (optional - be selective!)
    # ==========================================
    print("Logging training data...")
    mlflow.log_table(X, "X_train.parquet")
    mlflow.log_table(y, "y_train.parquet")

    # ==========================================
    # 6. LOG METADATA (same as before)
    # ==========================================
    print("Logging metadata...")
    meta_info = {
        "params": PARAMS,
        "data_info": {
            "n_samples": X.shape[0],
            "n_features": X.shape[1]
        },
        "notes": "Run sent from local to remote server"
    }
    with open("meta.json", "w") as f:
        json.dump(meta_info, f, indent=2)
    mlflow.log_artifact("meta.json")

    # ==========================================
    # 7. LOG PREPROCESSING CODE — CRITICAL FOR REMOTE 🆕
    # ==========================================
    # The remote server (Fink) needs your preprocessing code to transform
    # new incoming data the same way you transformed your training data.
    print("Logging preprocessing code...")
    mlflow.log_artifact("preprocessing.py", artifact_path="code")

    # ==========================================
    # 8. LOG REQUIREMENTS — CRITICAL FOR REMOTE 🆕
    # ==========================================
    # Tells the remote server which Python packages your preprocessing
    # code needs (MLflow handles the model's own dependencies automatically).
    print("Logging dependencies...")
    mlflow.log_artifact(dependencies_path)

    print("\nRun completed successfully on REMOTE server!")
    print(f"View at: {REMOTE_URI}")

## 5. Verify the Remote Run

Let's verify that everything was uploaded correctly:


In [ ]:
# Query the remote server
remote_experiment = client.get_experiment_by_name("tutorial")
remote_runs = client.search_runs(remote_experiment.experiment_id, max_results=5)

print("Latest runs on remote server:\n")

for i, run in enumerate(remote_runs[:3], 1):
    print(f"{i}. {run.data.tags.get('mlflow.runName', 'N/A')}")
    print(f"   - Run ID: {run.info.run_id[:8]}...")
    print(f"   - Accuracy: {run.data.metrics.get('accuracy', 'N/A')}")
    print(f"   - Status: {run.info.status}")
    print()

## 6. What We Sent vs. What We Didn't

### ✅ What we sent to the remote server:
- Trained model (sklearn artifact)
- Model parameters
- Performance metrics
- Training data (X and y) - optional
- Preprocessing code (`preprocessing.py`)
- Requirements.txt (dependencies)
- Metadata (custom info)

### 🙅🏿 What we didn't send:
- All the failed/experimental runs
- Intermediate debugging runs
- Large datasets from experimentation
- Temporary files

This keeps the remote server clean and focused on production-ready models!

---


###  <img src="https://media.giphy.com/media/v1.Y2lkPTc5MGI3NjExeXd2ZTNscDZtbm1jbDB2dDJveGp2ZWdtMmhsMDg0MmdvdnpncmJzZSZlcD12MV9zdGlja2Vyc19zZWFyY2gmY3Q9cw/xUPGchCSJlq3NzwLyU/giphy.gif" width="50" height="50"> WHY  Steps 7 & 8 Are Essential for Remote Deployment

When you send a model to the Fink server, you're not just sending the trained model weights. 
For the model to work in production, Fink needs:

1. **Your preprocessing code** (Step 7):
   - The server receives raw alert data
   - It needs to transform this data exactly as you did during training
   - Your `make_cut()`, `raw2clean()`, `run_sherlock()`, and `make_X()` functions
   - Without this, the model would receive incorrectly formatted data

2. **Your dependencies** (Step 8):
   - Your preprocessing code might use specific libraries (pandas, numpy, lasair, etc.)
   - The server needs to know which versions to install
   - Prevents runtime errors due to missing or incompatible packages

**Real-world scenario:**
- You train a model locally with your preprocessing pipeline
- Fink receives new alerts and needs to classify them
- Fink loads your model AND your preprocessing code
- For each alert: `raw data → your preprocessing → your model → prediction`
- Other scientists can also download and run your complete pipeline

**Without these steps**, your model would be useless on the remote server! 🚫



<iframe src="https://giphy.com/embed/S3nZ8V9uemShxiWX8g" width="40" height="40" style="" frameBorder="0" class="giphy-embed" allowFullScreen></iframe>Congratulations! You finished Tutorial 2!<iframe src="https://giphy.com/embed/8fftcK2D4PK6XCs2P0" width="60" height="60" style="" frameBorder="0" class="giphy-embed" allowFullScreen></iframe>

<!-- ---

## 🚀 Next Steps

In **Tutorial 3** ??? (optional), you'll learn how to:
<!-- - Customize the preprocessing pipeline for your data
- Replace the example model with your own
- Adapt this template for your specific project -->


## 🆘 Troubleshooting  <iframe src="https://giphy.com/embed/PnpkimJ5mrZRe" width="200" height="200" style="" frameBorder="0" class="giphy-embed" allowFullScreen></iframe>

### Problem: Authentication failed
**Solution**: Double-check your environment variables are set correctly. Try logging out and back in.

### Problem: Can't find local artifacts
**Solution**: Make sure `root_dir` points to where you ran `mlflow server`. Look for `mlruns/` and `mlartifacts/` folders.

### Problem: "Permission denied" on remote server
**Solution**: Verify your username has write access. Contact server administrator if needed.

### Problem: Missing preprocessing code on remote
**Solution**: Make sure you logged the preprocessing directory with `mlflow.log_artifacts()`

---
